# Create WecOptTool Model for the Marson WEC concept

In [1]:
import logging
import numpy as np
import capytaine as cpt
from capytaine.io import mesh_writers
import pygmsh
import gmsh
import vtk
import matplotlib.pyplot as plt
import os
import xarray as xr
from capytaine.io.meshio import load_from_meshio

logging.basicConfig(level=logging.INFO,format="%(levelnames)s:\t%(message)s")

#### WEC geometry mesh
Now we will create a surface mesh for the WEC hull and store it using the `FloatingBody` object from Capytaine.

In [2]:

# platform is a rectangle with cg at -0.46 and 1.44 m long, 0.76 m wide
flapThickness = 0.05

platformLength = 1.44 
outboardWidth = 0.1
platformWidth = 0.76+2*outboardWidth
platformHeight = 0.05
platformCG = [0, 0, -0.46]
columnsRadius = 0.05
columnsDraft = 0.6
columnsXY = [platformLength/2-flapThickness/2, platformWidth/2]
cutoutWidth = 0.76
cutoutLength = 1.44-2*outboardWidth

flapWidth = platformWidth-2*outboardWidth
flapHeight = 0.56
flap1CG = [-platformLength/2+flapThickness/2, 0, platformCG[2]+0.56/2] # from water surface/origin
flap2CG = [platformLength/2-flapThickness/2, 0, platformCG[2]+0.56/2] # from water surface/origin

In [3]:
#with pygmsh.geo.Geometry() as geom:
#    platform = geom.add_box(-platformLength/2, platformLength/2, -platformWidth/2, platformWidth/2, 
#                       -platformHeight/2, platformHeight/2, mesh_size = 0.15)
#    platformMesh = geom.generate_mesh()

with pygmsh.occ.Geometry() as geom:
    gmsh.option.setNumber('Mesh.MeshSizeFactor', 0.15)
    platform = geom.add_box([-platformLength/2, -platformWidth/2, -platformHeight/2], [platformLength,platformWidth,platformHeight])
    cutout = geom.add_box([-cutoutLength/2, -cutoutWidth/2, -platformHeight/2], [cutoutLength,cutoutWidth,platformHeight])
    cyl1 = geom.add_cylinder([-columnsXY[0], -columnsXY[1], -columnsDraft-platformCG[2]],[0, 0, columnsDraft+.01], columnsRadius)
    cyl2 = geom.add_cylinder([-columnsXY[0], columnsXY[1], -columnsDraft-platformCG[2]],[0, 0, columnsDraft+.01], columnsRadius)
    cyl3 = geom.add_cylinder([columnsXY[0], -columnsXY[1], -columnsDraft-platformCG[2]],[0, 0, columnsDraft+.01], columnsRadius)
    cyl4 = geom.add_cylinder([columnsXY[0], columnsXY[1], -columnsDraft-platformCG[2]],[0, 0, columnsDraft+.01], columnsRadius)
    platformCutout = geom.boolean_difference(platform,cutout)
    geom.boolean_union([platformCutout,cyl1,cyl2,cyl3,cyl4])
    platformMesh = geom.generate_mesh()

fb = cpt.FloatingBody(mesh=platformMesh, name=f'platform', center_of_mass=[0,0,0])

cpt.io.mesh_writers.write_STL('platform.stl',fb.mesh.vertices, fb.mesh.faces)

In [14]:
with pygmsh.geo.Geometry() as geom:
    flap = geom.add_box(-flapThickness/2, flapThickness/2, -flapWidth/2, flapWidth/2, 
                       -flapHeight/2, flapHeight/2, mesh_size = 0.15)
    flapMesh = geom.generate_mesh()

fb = cpt.FloatingBody(mesh=flapMesh, name=f'flap', center_of_mass=[0,0,0])

cpt.io.mesh_writers.write_STL('flap.stl',fb.mesh.vertices, fb.mesh.faces)